# 01 - Data Audit

Scan dữ liệu gốc: cấu trúc, kiểu dữ liệu, missing, duplicate, range, outlier, consistency. Notebook này tạo nền tảng cho cleaning và EDA.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Create necessary directories
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

# Load dataset and basic audit
title("Load dataset")
df = clean_cols(pd.read_csv(RAW_PATH))
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")
display(df.head())

# Basic audit and data types
title("Basic audit")
audit = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate rows", "Total missing values", "Memory MB"],
    "Value": [df.shape[0], df.shape[1], df.duplicated().sum(), df.isna().sum().sum(), round(df.memory_usage(deep=True).sum()/1024**2, 3)]
})
dtypes = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique().values,
    "Missing Count": df.isna().sum().values,
    "Missing %": (df.isna().mean().values * 100).round(2)
})
display(audit); display(dtypes)



Load dataset
Rows: 1,000 | Columns: 10


,User_ID,Age,Gender,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Location
0,1,56,Male,2.61,7.15,24,4.43,0.55,2.40,Los Angeles
1,2,46,Male,2.13,13.79,18,4.67,4.42,2.43,Chicago
2,3,32,Female,7.28,4.50,11,4.58,1.71,2.83,Houston
3,4,25,Female,1.20,6.29,21,3.18,3.42,4.58,Phoenix
4,5,38,Male,6.31,12.59,14,3.15,0.13,4.00,New York



Basic audit


,Metric,Value
0,Rows,1000.000
1,Columns,10.000
2,Duplicate rows,0.000
3,Total missing values,0.000
4,Memory MB,0.167


,Column,Data Type,Unique Values,Missing Count,Missing %
0,User_ID,int64,1000,0,0.0
1,Age,int64,42,0,0.0
2,Gender,object,2,0,0.0
3,Total_App_Usage_Hours,float64,672,0,0.0
4,Daily_Screen_Time_Hours,float64,705,0,0.0
5,Number_of_Apps_Used,int64,27,0,0.0
6,Social_Media_Usage_Hours,float64,437,0,0.0
7,Productivity_App_Usage_Hours,float64,432,0,0.0
8,Gaming_App_Usage_Hours,float64,443,0,0.0
9,Location,object,5,0,0.0


In [2]:
# Numerical and categorical summary
title("Numerical and categorical summary")
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

# Numerical summary
display(df[num_cols].agg(["count", "mean", "median", "std", "min", "max", "skew"]).T.round(3))
for c in cat_cols:
    print(f"\n{c}")
    display(df[c].value_counts().to_frame("count"))



Numerical and categorical summary
Numerical: ['User_ID', 'Age', 'Total_App_Usage_Hours', 'Daily_Screen_Time_Hours', 'Number_of_Apps_Used', 'Social_Media_Usage_Hours', 'Productivity_App_Usage_Hours', 'Gaming_App_Usage_Hours']
Categorical: ['Gender', 'Location']


,count,mean,median,std,min,max,skew
User_ID,1000.0,500.500,500.500,288.819,1.00,1000.00,0.000
Age,1000.0,38.745,40.000,12.187,18.00,59.00,-0.089
Total_App_Usage_Hours,1000.0,6.406,6.455,3.135,1.00,11.97,0.025
Daily_Screen_Time_Hours,1000.0,7.696,7.880,3.714,1.01,14.00,-0.107
Number_of_Apps_Used,1000.0,16.647,17.000,7.620,3.00,29.00,-0.081
Social_Media_Usage_Hours,1000.0,2.456,2.445,1.440,0.00,4.99,0.041
Productivity_App_Usage_Hours,1000.0,2.495,2.435,1.443,0.00,5.00,0.036
Gaming_App_Usage_Hours,1000.0,2.475,2.455,1.450,0.01,5.00,0.046



Gender


,count
Gender,
Male,517
Female,483



Location


,count
Location,
New York,243
Phoenix,199
Chicago,192
Los Angeles,185
Houston,181


In [3]:
# Range, outlier, and distribution audit
title("Range, outlier, and distribution audit")
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((df[c] < lo) | (df[c] > hi)).sum()
    rows.append({"Column": c, "Min": df[c].min(), "Q1": q1, "Median": df[c].median(), "Q3": q3, "Max": df[c].max(), "Outliers": count, "Outlier %": round(count/len(df)*100, 2)})
outliers = pd.DataFrame(rows)
display(outliers)

# Boxplots and histograms
long_num = df.melt(value_vars=num_cols, var_name="Metric", value_name="Value")
fig = px.box(long_num, x="Metric", y="Value", color="Metric", title="Boxplot audit for numerical variables")
fig.update_layout(showlegend=False, xaxis_tickangle=-35, height=520)
fig.show()

# Histograms with box margins
for c in num_cols:
    fig = px.histogram(df, x=c, nbins=30, marginal="box", title=f"Distribution — {c}")
    fig.update_layout(height=420)
    fig.show()



Range, outlier, and distribution audit


,Column,Min,Q1,Median,Q3,Max,Outliers,Outlier %
0,User_ID,1.00,250.7500,500.500,750.2500,1000.00,0,0.0
1,Age,18.00,28.0000,40.000,50.0000,59.00,0,0.0
2,Total_App_Usage_Hours,1.00,3.5900,6.455,9.1225,11.97,0,0.0
3,Daily_Screen_Time_Hours,1.01,4.5300,7.880,10.9100,14.00,0,0.0
4,Number_of_Apps_Used,3.00,10.0000,17.000,23.0000,29.00,0,0.0
5,Social_Media_Usage_Hours,0.00,1.2000,2.445,3.6725,4.99,0,0.0
6,Productivity_App_Usage_Hours,0.00,1.2825,2.435,3.7100,5.00,0,0.0
7,Gaming_App_Usage_Hours,0.01,1.2200,2.455,3.7825,5.00,0,0.0


In [4]:
# Data consistency check
title("Data consistency check")
required = ["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours"]
if all(c in df.columns for c in required):
    df["Category_Usage_Sum"] = df["Social_Media_Usage_Hours"] + df["Productivity_App_Usage_Hours"] + df["Gaming_App_Usage_Hours"]
    df["Category_vs_Total_Diff"] = df["Category_Usage_Sum"] - df["Total_App_Usage_Hours"]
    display(df[["Total_App_Usage_Hours", "Category_Usage_Sum", "Category_vs_Total_Diff"]].describe().round(3))
    inconsistent = (df["Category_Usage_Sum"] > df["Total_App_Usage_Hours"]).sum()
    print(f"Rows where category sum > total app usage: {inconsistent:,} ({inconsistent/len(df)*100:.2f}%)")
    fig = px.histogram(df, x="Category_vs_Total_Diff", nbins=40, title="Category usage sum - total app usage")
    fig.show()
    print("Interpretation note: category columns should be treated as behavioral intensity indicators, not exact parts of total usage.")



Data consistency check


,Total_App_Usage_Hours,Category_Usage_Sum,Category_vs_Total_Diff
count,1000.000,1000.000,1000.000
mean,6.406,7.427,1.021
std,3.135,2.466,4.055
min,1.000,0.730,-8.690
25%,3.590,5.710,-1.873
50%,6.455,7.400,0.935
75%,9.122,9.075,3.945
max,11.970,14.240,12.370


Rows where category sum > total app usage: 588 (58.80%)


Interpretation note: category columns should be treated as behavioral intensity indicators, not exact parts of total usage.


In [8]:
# Correlation and first relationships
title("Correlation and first relationships")
corr = df[num_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True, aspect="auto", title="Correlation matrix")
fig.show()

# Scatter relationships with screen time
pairs = [("Number_of_Apps_Used", "Daily_Screen_Time_Hours"), ("Total_App_Usage_Hours", "Daily_Screen_Time_Hours"), ("Social_Media_Usage_Hours", "Daily_Screen_Time_Hours"), ("Productivity_App_Usage_Hours", "Daily_Screen_Time_Hours"), ("Gaming_App_Usage_Hours", "Daily_Screen_Time_Hours")]
for x, y in pairs:
    if x in df.columns and y in df.columns:
        fig = px.scatter(df, x=x, y=y, opacity=0.55, trendline="ols", title=f"{x} vs {y}")
        fig.update_layout(height=470)
        fig.show()



Correlation and first relationships


In [7]:
# Save audit preview
title("Save audit preview")
out = PROCESSED_DIR / "smartphone_audit_preview.csv"
df.to_csv(out, index=False)
print(f"Saved: {out}")
print("Key audit observations:")
print(f"- Rows: {df.shape[0]:,}; columns: {df.shape[1]:,}")
print(f"- Missing values: {df.isna().sum().sum():,}; duplicates: {df.duplicated().sum():,}")
print("- Category usage sum is not guaranteed to equal Total_App_Usage_Hours.")



Save audit preview
Saved: ../data/processed/smartphone_audit_preview.csv
Key audit observations:
- Rows: 1,000; columns: 12
- Missing values: 0; duplicates: 0
- Category usage sum is not guaranteed to equal Total_App_Usage_Hours.
